# Final Comparative Analysis — LLMs × Prompt Strategies for Code-Smell Detection

**Author:** Senior research analysis · **Dataset:** `prepared_data/datasets/annotated` (34 source files, 4 languages, 473 line-level smell annotations)
**Models evaluated** (AWS Bedrock, `ap-southeast-2`, `temperature=0`, `seed=42`):

| Model | Param scale | Family |
|---|---|---|
| `deepseek.v3.2` | mixture-of-experts | DeepSeek |
| `google.gemma-3-27b-it` | 27 B | Gemma |
| `mistral.devstral-2-123b` | 123 B | Mistral Devstral |
| `qwen.qwen3-coder-30b-a3b-v1` | 30 B | Qwen (coder) |
| `openai.gpt-oss-120b-1` | 120 B | OpenAI (OSS) |
| `google.gemma-3-4b-it` | 4 B | Gemma |
| `mistral.voxtral-mini-3b-2507` | 3 B | Mistral |
| `nvidia.nemotron-nano-3-30b` | 3 B | NVIDIA Nemotron |
| `nvidia.nemotron-nano-9b-v2` | 9 B | NVIDIA Nemotron |

**Prompt strategies (RQ1 → RQ3):**

| ID | Strategy | RQ |
|---|---|---|
| P1 | Zero-shot (taxonomy in system prompt) | RQ1 — base capability |
| P2 | Few-shot (3 in-context examples) | RQ1 — example effect |
| P3 | Taxonomy-tree CoT (5 categories → smells) | RQ2 — structured reasoning |
| P4 | Self-verify (predict → critique → revise) | RQ2 — self-correction |
| P5 dense (k=3) | RAG with dense embeddings | RQ3 — retrieval grounding |
| P5 random (k=2) | RAG ablation with random retrieval | RQ3 — retrieval ablation |

This notebook produces the **paper-ready comparative analysis**:

1. Setup & metrics ingestion (all models × all prompts × all languages)
2. Headline table — micro/macro F1 per model × prompt
3. Best strategy per model (and per language)
4. Cross-model leaderboard with bootstrap CIs
5. Prompt-strategy effect (Δ over zero-shot baseline)
6. RAG ablation: dense (k=3) vs random (k=2)
7. Per-language sensitivity heatmaps
8. Per-smell macro-F1 — where each model wins/loses
9. Error mix: parse errors, FP vs FN, hallucinated smells
10. Cost / latency vs F1 trade-off
11. Statistical significance (paired bootstrap on per-record F1)
12. Discussion & paper-ready key findings


In [ ]:
from __future__ import annotations

import json, re
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT     = Path.cwd()
while not (ROOT / "results" / "llm_runs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RUNS_DIR = ROOT / "results" / "llm_runs"
FIG_DIR  = ROOT / "results" / "figures" / "final_comparative"
TBL_DIR  = ROOT / "results" / "tables"   / "final_comparative"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TBL_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 300
plt.rcParams.update({
    "font.size": 13,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
})
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

MODELS = {
    "deepseekv3_2"            : "DeepSeek v3.2",
    "gemma_3_27b_it"          : "Gemma 3 27B-IT",
    "mistral_devstral_2_123b" : "Mistral Devstral 2 123B",
    "qwen.qwen3-coder-30b-a3b-v1"    : "Qwen3 Coder 30B A3B",
    "openai.gpt-oss-120b-1"          : "GPT-OSS 120B",
    "google.gemma-3-4b-it"           : "Gemma 3 4B-IT",
    "mistral.voxtral-mini-3b-2507"   : "Mistral Voxtral Mini 3B",
    "nvidia.nemotron-nano-3-30b"     : "Nemotron Nano 3-30B",
    "nvidia.nemotron-nano-9b-v2"     : "Nemotron Nano 9B v2",
}
MODEL_PAL = {
    "DeepSeek v3.2"           : "#1f77b4",
    "Gemma 3 27B-IT"          : "#ff7f0e",
    "Mistral Devstral 2 123B" : "#2ca02c",
    "Qwen3 Coder 30B A3B"     : "#d62728",
    "GPT-OSS 120B"            : "#9467bd",
    "Gemma 3 4B-IT"           : "#8c564b",
    "Mistral Voxtral Mini 3B" : "#e377c2",
    "Nemotron Nano 3-30B"     : "#bcbd22",
    "Nemotron Nano 9B v2"     : "#17becf",
}

PROMPTS = [
    ("p1_zero_shot",      "P1 zero-shot"),
    ("p2_few_shot",       "P2 few-shot"),
    ("p3_taxonomy_tree",  "P3 tax-tree"),
    ("p4_self_verify",    "P4 self-verify"),
    ("p5_rag_dense",      "P5 RAG dense (k=3)"),
    ("p5_rag_random",     "P5 RAG random (k=2)"),
]
PROMPT_LABEL = dict(PROMPTS)
PROMPT_PAL   = dict(zip([p for p, _ in PROMPTS],
                        ["#4c72b0", "#dd8452", "#55a467", "#c44e52",
                         "#8172b3", "#937860"]))

PROMPT_SOURCE = {
    "p1_zero_shot"     : ("p1_zero_shot",     None),
    "p2_few_shot"      : ("p2_few_shot",      None),
    "p3_taxonomy_tree" : ("p3_taxonomy_tree", None),
    "p4_self_verify"   : ("p4_self_verify",   None),
    "p5_rag_dense"     : ("p5_rag",           "dense"),
    "p5_rag_random"    : ("p5_rag",           "random"),
}

LANGS = ["java", "python", "javascript", "cpp"]

print("ROOT     :", ROOT)
print("RUNS_DIR :", RUNS_DIR, "exists" if RUNS_DIR.exists() else "MISSING")
for m in MODELS:
    print(f"  {m:25s} ->", (RUNS_DIR / m).exists())

## 1 · Ingest all metrics files

For each `(model, prompt, language)` cell we keep the most recent metrics file.
P5 RAG is split into two virtual prompts by `meta.rag_mode`.

In [ ]:
def latest_metrics(model_dir: str, prompt_virtual: str, language: str) -> dict | None:
    folder, rag_mode = PROMPT_SOURCE[prompt_virtual]
    d = RUNS_DIR / model_dir / folder
    if not d.exists():
        return None
    pat = re.compile(rf"^{folder}__.+__annotated__{language}__all__.*\.metrics\.json$")
    candidates = sorted(p for p in d.iterdir() if pat.match(p.name))
    if rag_mode is not None:
        candidates = [p for p in candidates if f"__{rag_mode}_" in p.name]
    if not candidates:
        return None
    candidates.sort(key=lambda p: p.stat().st_mtime)
    return json.loads(candidates[-1].read_text())

rows = []
missing = []
for model_dir, model_label in MODELS.items():
    for prompt_v, prompt_label in PROMPTS:
        for lang in LANGS:
            m = latest_metrics(model_dir, prompt_v, lang)
            if m is None:
                missing.append((model_label, prompt_v, lang))
                continue
            ov   = m["overall"]
            meta = m.get("meta", {})
            tu   = m.get("token_usage", {}) or {}
            ci   = (m.get("bootstrap_ci") or {}).get("micro_f1") or {}
            rows.append({
                "model"          : model_label,
                "model_dir"      : model_dir,
                "prompt"         : prompt_v,
                "prompt_label"   : prompt_label,
                "language"       : lang,
                "precision"      : ov.get("micro_precision", ov.get("precision")),
                "recall"         : ov.get("micro_recall",    ov.get("recall")),
                "f1"             : ov.get("micro_f1",        ov.get("f1")),
                "macro_f1"       : ov.get("macro_f1"),
                "weighted_f1"    : ov.get("weighted_f1"),
                "tp"             : ov.get("tp", 0),
                "fp"             : ov.get("fp", 0),
                "fn"             : ov.get("fn", 0),
                "n_records"      : ov.get("n_records"),
                "parse_errors"   : ov.get("parse_errors", 0),
                "invalid"        : ov.get("invalid_findings", 0),
                "total_findings" : ov.get("total_findings", 0),
                "f1_lo"          : ci.get("lo", ci.get("lower")),
                "f1_hi"          : ci.get("hi", ci.get("upper")),
                "elapsed_sec"    : meta.get("elapsed_sec"),
                "rag_mode"       : meta.get("rag_mode"),
                "rag_k"          : meta.get("rag_k"),
                "tokens_in"      : tu.get("total_input_tokens", tu.get("prompt_tokens")),
                "tokens_out"     : tu.get("total_output_tokens", tu.get("completion_tokens")),
                "tokens_total"   : tu.get("total_tokens"),
                "timestamp"      : meta.get("timestamp"),
            })

df = pd.DataFrame(rows)
df["model"]    = pd.Categorical(df["model"], categories=list(MODELS.values()), ordered=True)
df["prompt"]   = pd.Categorical(df["prompt"], categories=[p for p, _ in PROMPTS], ordered=True)
df["language"] = pd.Categorical(df["language"], categories=LANGS, ordered=True)

print(f"Loaded {len(df)} (model × prompt × language) rows.")
if missing:
    print(f"\nMissing combinations ({len(missing)}):")
    for m_, p_, l_ in missing:
        print(f"  - {m_:30s} {p_:20s} {l_}")
df.head(10)

## 2 · Headline table — micro & macro F1 per model × prompt

We aggregate across the four languages by recomputing **micro** F1 from pooled TP/FP/FN
(the right thing to do statistically — averaging F1s would over-weight C++'s smaller GT set).

In [ ]:
def micro_from_counts(g):
    tp, fp, fn = g["tp"].sum(), g["fp"].sum(), g["fn"].sum()
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return pd.Series({"precision": p, "recall": r, "f1": f,
                      "tp": tp, "fp": fp, "fn": fn,
                      "macro_f1": g["macro_f1"].mean(),
                      "weighted_f1": g["weighted_f1"].mean()})

headline = (df.groupby(["model", "prompt"], observed=True)
              .apply(micro_from_counts, include_groups=False)
              .round(4))
headline_disp = headline[["precision", "recall", "f1", "macro_f1"]].rename(
    columns={"precision": "P", "recall": "R", "f1": "micro F1", "macro_f1": "macro F1"}
)
display(headline_disp)
headline_disp.to_csv(TBL_DIR / "01_headline_micro_macro.csv")

f1_pivot = headline["f1"].unstack("prompt").reindex(columns=[p for p, _ in PROMPTS])
display(f1_pivot.round(3).style.background_gradient(cmap="Greens", axis=None)
                              .format("{:.3f}").set_caption("micro F1 — model × prompt"))


In [ ]:
prompt_labels_short = [PROMPT_LABEL[p] for p in f1_pivot.columns]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.6))

hh = headline["f1"].reset_index()
sns.barplot(data=hh, x="prompt", y="f1", hue="model", palette=MODEL_PAL, ax=axes[0])
axes[0].set_title("Micro F1 per (model × prompt) — pooled across languages")
axes[0].set_ylabel("micro F1"); axes[0].set_xlabel("")
axes[0].set_xticklabels(prompt_labels_short, rotation=20, ha="right")
axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3, fontsize=9, frameon=False)
for c in axes[0].containers:
    axes[0].bar_label(c, fmt="%.2f", fontsize=8, padding=2)

sns.heatmap(f1_pivot, annot=True, fmt=".3f", cmap="YlGnBu",
            cbar_kws={"label": "micro F1"}, ax=axes[1])
axes[1].set_title("Heatmap — micro F1")
axes[1].set_xticklabels(prompt_labels_short, rotation=20, ha="right")
axes[1].set_xlabel(""); axes[1].set_ylabel("")

fig.tight_layout()
fig.savefig(FIG_DIR / "02_headline_f1.png", bbox_inches="tight")
plt.show()

## 3 · Best strategy per model — and overall winner

In [ ]:
best = (f1_pivot.idxmax(axis=1).rename("best_prompt").to_frame()
        .assign(best_f1=f1_pivot.max(axis=1).round(3),
                worst_prompt=f1_pivot.idxmin(axis=1),
                worst_f1=f1_pivot.min(axis=1).round(3),
                spread=(f1_pivot.max(axis=1) - f1_pivot.min(axis=1)).round(3)))
best["best_prompt"]  = best["best_prompt"].map(PROMPT_LABEL)
best["worst_prompt"] = best["worst_prompt"].map(PROMPT_LABEL)
display(best)

overall_best = headline["f1"].idxmax()
print(f"\n🏆 Overall best: {overall_best[0]} with {PROMPT_LABEL[overall_best[1]]} "
      f"(micro F1 = {headline['f1'].max():.3f})")

## 4 · Cross-model leaderboard with bootstrap CIs

We use the bootstrap CI on micro F1 emitted by each per-language run, aggregated by
prompt-strategy (mean lower / upper across the four languages — a conservative pooled CI).

In [ ]:
ci = (df.groupby(["model", "prompt"], observed=True)[["f1", "f1_lo", "f1_hi"]]
        .mean().round(4))
ci.columns = ["f1 (mean of langs)", "CI low", "CI high"]
display(ci)

plot_df = ci.reset_index().rename(columns={"f1 (mean of langs)": "f1"})
plot_df["err_low"]  = plot_df["f1"] - plot_df["CI low"]
plot_df["err_high"] = plot_df["CI high"] - plot_df["f1"]

fig, ax = plt.subplots(figsize=(11, 4.8))
n_models = len(MODELS)
x = np.arange(len(PROMPTS))
w = 0.8 / n_models
for i, (m_label) in enumerate(MODELS.values()):
    sub = plot_df[plot_df.model == m_label].set_index("prompt").reindex([p for p, _ in PROMPTS])
    ax.bar(x + (i - (n_models - 1) / 2) * w, sub.f1, width=w,
           yerr=[sub.err_low.fillna(0), sub.err_high.fillna(0)],
           capsize=3, label=m_label, color=MODEL_PAL[m_label], edgecolor="black", linewidth=0.4)
ax.set_xticks(x); ax.set_xticklabels(prompt_labels_short, rotation=20, ha="right")
ax.set_ylabel("micro F1 (mean across languages, ±95% bootstrap CI)")
ax.set_title("Leaderboard with bootstrap CIs")
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "03_leaderboard_ci.png", bbox_inches="tight")
plt.show()

## 5 · Prompt-strategy effect: Δ over zero-shot baseline

For each model, P1 is the baseline. We report **ΔF1 = F1(strategy) − F1(P1)**.
Positive Δ ⇒ the strategy improves over zero-shot.

In [ ]:
delta = f1_pivot.subtract(f1_pivot["p1_zero_shot"], axis=0).round(3)
delta_disp = delta.copy()
delta_disp.columns = [PROMPT_LABEL[c] for c in delta_disp.columns]
display(delta_disp.style.background_gradient(cmap="RdYlGn", axis=None, vmin=-0.2, vmax=0.2)
                  .format("{:+.3f}").set_caption("ΔF1 vs P1 zero-shot"))

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.heatmap(delta, annot=True, fmt="+.3f", cmap="RdYlGn", center=0,
            cbar_kws={"label": "ΔF1 vs P1"}, ax=ax,
            xticklabels=prompt_labels_short)
ax.set_title("Prompt-strategy lift over zero-shot baseline")
ax.set_xlabel(""); ax.set_ylabel("")
plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
fig.tight_layout()
fig.savefig(FIG_DIR / "04_delta_vs_p1.png", bbox_inches="tight")
plt.show()


## 6 · RAG ablation — dense (k=3) vs random (k=2)

If retrieval **content** matters, dense should beat random by a large margin.
If only the *presence* of extra context matters, the gap will be small.

In [ ]:
rag = df[df.prompt.isin(["p5_rag_dense", "p5_rag_random"])].copy()
rag_pivot = rag.pivot_table(index=["model", "language"], columns="prompt",
                            values="f1", observed=True)
rag_pivot["Δ (dense − random)"] = (rag_pivot["p5_rag_dense"] -
                                    rag_pivot["p5_rag_random"]).round(3)
rag_pivot = rag_pivot.rename(columns={"p5_rag_dense": "dense (k=3)",
                                       "p5_rag_random": "random (k=2)"})
display(rag_pivot.round(3))
rag_pivot.to_csv(TBL_DIR / "02_rag_ablation_per_lang.csv")

rag_pooled = (rag.groupby(["model", "prompt"], observed=True)
                 .apply(micro_from_counts, include_groups=False)["f1"]
                 .unstack("prompt").round(3))
rag_pooled["Δ"] = (rag_pooled["p5_rag_dense"] - rag_pooled["p5_rag_random"]).round(3)
rag_pooled = rag_pooled.rename(columns={"p5_rag_dense": "dense", "p5_rag_random": "random"})
print("\nPooled across languages (micro F1):")
display(rag_pooled)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.6), gridspec_kw={"width_ratios": [1.2, 1]})
ax = axes[0]
ax2 = ax.twinx()
mlabels4 = [m for m in MODELS.values()
            if m in rag_pooled.index and pd.notna(rag_pooled.loc[m, "random"])]
xs4 = np.arange(len(mlabels4))
dense4  = rag_pooled.loc[mlabels4, "dense"]
random4 = rag_pooled.loc[mlabels4, "random"]
delta4  = (dense4 - random4).round(3)
ax.bar(xs4 - 0.2, dense4,  width=0.4, label="dense (k=3)",  color="#2a9d8f", edgecolor="black", linewidth=0.4)
ax.bar(xs4 + 0.2, random4, width=0.4, label="random (k=2)",
       color="#e9c46a", edgecolor="black", linewidth=0.4)
ax2.plot(xs4, delta4, "o-", color="#c44e52", linewidth=2, markersize=8, label="Δ (right axis)")
ax2.axhline(0, color="grey", linewidth=0.6, linestyle="--")
ax.set_xticks(xs4); ax.set_xticklabels(mlabels4, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("micro F1"); ax2.set_ylabel("Δ (dense − random)")
ax.set_title("Dense vs random retrieval (content effect)")
ax.legend(loc="upper left"); ax2.legend(loc="upper right")

bx = axes[1]
lift9 = (f1_pivot["p5_rag_dense"] - f1_pivot["p1_zero_shot"]).reindex(list(MODELS.values()))
xs9 = np.arange(len(lift9))
colors9 = ["#2a9d8f" if v >= 0 else "#c44e52" for v in lift9.values]
bx.bar(xs9, lift9.values, color=colors9, edgecolor="black", linewidth=0.4)
bx.axhline(0, color="grey", linewidth=0.8)
bx.set_xticks(xs9); bx.set_xticklabels(list(MODELS.values()), rotation=30, ha="right", fontsize=9)
bx.set_ylabel("Δ micro F1 (dense − P1)")
bx.set_title("Dense RAG lift over zero-shot (all 9 models)")
for x, v in zip(xs9, lift9.values):
    bx.text(x, v + (0.008 if v >= 0 else -0.022), f"{v:+.3f}", ha="center", fontsize=7)
fig.suptitle("RAG ablation — retrieval content and retrieval lift", fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "05_rag_ablation.png", bbox_inches="tight")
plt.show()

## 7 · Per-language sensitivity heatmaps

Some smells are heavy in C++ (Bloaters dominate after method extraction) while
Java/Python carry most Change-Preventer labels. Models may behave very differently per language.

In [ ]:
model_order = list(MODELS.values())
fig, axes = plt.subplots(3, 3, figsize=(13, 12.5))
for i, m_label in enumerate(model_order):
    r, c = divmod(i, 3)
    ax = axes[r, c]
    sub = (df[df.model == m_label]
             .pivot_table(index="prompt", columns="language", values="f1", observed=True)
             .reindex([p for p, _ in PROMPTS])[LANGS])
    sns.heatmap(sub, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1,
                cbar=(c == 2), ax=ax, annot_kws={"fontsize": 11},
                cbar_kws={"label": "micro F1"} if c == 2 else None)
    ax.set_title(m_label, fontsize=13, fontweight="bold")
    ax.set_xlabel(""); ax.set_ylabel("")
    ax.set_yticklabels([PROMPT_LABEL[p] for p in sub.index], rotation=0, fontsize=10)
    lang_short = {"java": "Java", "python": "Python", "javascript": "JS", "cpp": "C++"}
    ax.set_xticklabels([lang_short[l] for l in LANGS], rotation=20, ha="right", fontsize=10)
fig.suptitle("Micro F1 \u2014 language sensitivity per model", y=1.0, fontsize=15)
fig.tight_layout()
fig.savefig(FIG_DIR / "06_per_language_heatmaps.png", bbox_inches="tight")
plt.show()

lang_mean = df.groupby("language", observed=True)["f1"].agg(["mean", "std", "min", "max"]).round(3)
print("\nLanguage difficulty (mean F1 across all model × prompt cells):")
display(lang_mean)

## 8 · Per-smell macro analysis — where each model wins / loses

Re-load `per_smell` blocks from each metrics file (best-prompt per model) and pool TP/FP/FN
across the four languages to compute a **per-smell macro F1**.

In [ ]:
def per_smell_pooled(model_dir: str, prompt_v: str) -> pd.DataFrame:
    rows_ = []
    for lang in LANGS:
        m = latest_metrics(model_dir, prompt_v, lang)
        if not m:
            continue
        for smell, st in (m.get("per_smell") or {}).items():
            rows_.append({
                "smell": smell,
                "tp": st.get("tp", 0),
                "fp": st.get("fp", 0),
                "fn": st.get("fn", 0),
            })
    if not rows_:
        return pd.DataFrame()
    sm = pd.DataFrame(rows_).groupby("smell")[["tp", "fp", "fn"]].sum()
    sm["precision"] = sm.tp / (sm.tp + sm.fp).replace(0, np.nan)
    sm["recall"]    = sm.tp / (sm.tp + sm.fn).replace(0, np.nan)
    sm["f1"] = 2 * sm.precision * sm.recall / (sm.precision + sm.recall)
    sm = sm.fillna(0.0)
    sm["support"] = sm.tp + sm.fn
    return sm[sm.support > 0]

best_prompt_per_model = {m: f1_pivot.loc[m].idxmax() for m in MODELS.values()}
print("Best prompt per model:")
for k, v in best_prompt_per_model.items():
    print(f"  {k:30s}: {PROMPT_LABEL[v]}")

per_smell_all = {}
for model_dir, m_label in MODELS.items():
    p = best_prompt_per_model[m_label]
    sm = per_smell_pooled(model_dir, p)
    sm = sm.assign(model=m_label, prompt=p)
    per_smell_all[m_label] = sm

ps_f1 = pd.concat([d.assign(model=k)[["model", "f1", "support"]].reset_index()
                   for k, d in per_smell_all.items()])
ps_pivot = (ps_f1.pivot_table(index="smell", columns="model", values="f1")
                  .reindex(columns=list(MODELS.values())))
support = ps_f1.groupby("smell")["support"].max()
ps_pivot["support"] = support
ps_pivot = ps_pivot[ps_pivot["support"] > 0].sort_values("support", ascending=False)
display(ps_pivot.round(3).style.background_gradient(
    cmap="YlGnBu", subset=list(MODELS.values()), axis=None
).format("{:.2f}", subset=list(MODELS.values())).format("{:.0f}", subset=["support"]))
ps_pivot.to_csv(TBL_DIR / "03_per_smell_f1_best_prompt.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 0.40 * len(ps_pivot) + 2))
ps_pivot[list(MODELS.values())].plot(kind="barh", ax=ax,
                                       color=[MODEL_PAL[m] for m in MODELS.values()],
                                       edgecolor="black", linewidth=0.3, width=0.85)
ax.invert_yaxis()
ax.set_xlabel("F1 (best prompt per model, pooled across languages)")
ax.set_title("Per-smell F1 \u2014 winners and weaknesses")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.03 + 0.004 * len(ps_pivot)),
          ncol=3, fontsize=10, frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "07_per_smell_f1.png", bbox_inches="tight")
plt.show()

gap = (ps_pivot[list(MODELS.values())].max(axis=1) -
       ps_pivot[list(MODELS.values())].min(axis=1)).rename("gap")
display(pd.concat([ps_pivot[list(MODELS.values())], gap, ps_pivot["support"]], axis=1)
        .sort_values("gap", ascending=False).head(10).round(3))

## 9 · Error mix — FP vs FN balance & parsing failures

A model with high FP is **over-suggestive** (cries wolf); high FN is **conservative** (misses smells).
Parse errors and invalid findings flag JSON-output reliability — important for downstream tooling.

In [ ]:
err = (df.groupby(["model", "prompt"], observed=True)[
        ["tp", "fp", "fn", "parse_errors", "invalid", "total_findings"]]
       .sum())
err["FP/FN"]    = (err.fp / err.fn.replace(0, np.nan)).round(2)
err["err_rate"] = ((err.parse_errors + err.invalid) /
                    err.total_findings.replace(0, np.nan)).round(3)
display(err)
err.to_csv(TBL_DIR / "04_error_mix.csv")

prompt_order = [p for p, _ in PROMPTS]
model_order = list(MODELS.values())
fig = plt.figure(figsize=(19, 11))
gs = fig.add_gridspec(3, 4, width_ratios=[1, 1, 1, 0.05], wspace=1.15, hspace=0.8)
kind_colors = {"tp": "#1f77b4", "fp": "#d62728", "fn": "#ff7f0e"}
for i, m_label in enumerate(model_order):
    r, c = divmod(i, 3)
    ax = fig.add_subplot(gs[r, c])
    sub = err.loc[m_label].reindex(prompt_order)[["tp", "fp", "fn"]] if m_label in err.index.get_level_values(0) else None
    if sub is None or sub.empty:
        ax.axis("off")
        continue
    left = np.zeros(len(sub))
    for kind in ["tp", "fp", "fn"]:
        vals = sub[kind].fillna(0).values
        ax.barh(range(len(sub)), vals, left=left, color=kind_colors[kind], label=kind,
                edgecolor="black", linewidth=0.2)
        left = left + vals
    ax.set_yticks(range(len(sub)))
    ax.set_yticklabels(prompt_labels_short, fontsize=8)
    ax.invert_yaxis()
    ax.set_title(m_label, fontsize=10, fontweight="bold")
    ax.tick_params(axis="x", labelsize=8)
    if c == 0:
        ax.set_xlabel("count", fontsize=8)
handles = [plt.Rectangle((0, 0), 1, 1, color=kind_colors[k]) for k in ["tp", "fp", "fn"]]
fig.legend(handles, ["TP", "FP", "FN"], loc="upper center", ncol=3,
           bbox_to_anchor=(0.5, 1.02), fontsize=11, frameon=False)
fig.suptitle("TP / FP / FN per (model, prompt), faceted by model", y=1.06, fontsize=13)
fig.savefig(FIG_DIR / "08_error_mix.png", bbox_inches="tight")
plt.show()

fig2, ax2 = plt.subplots(figsize=(12, 5.6))
err_rate = err["err_rate"].unstack("prompt").reindex(columns=prompt_order).reindex(model_order)
sns.heatmap(err_rate.fillna(0), annot=True, fmt=".3f", cmap="OrRd",
            cbar_kws={"label": "(parse_err + invalid) / findings"}, ax=ax2,
            xticklabels=prompt_labels_short, annot_kws={"fontsize": 11})
ax2.set_title("Output-format error rate", fontsize=14)
ax2.set_xlabel(""); ax2.set_ylabel("")
plt.setp(ax2.get_xticklabels(), rotation=45, ha="right", fontsize=11)
plt.setp(ax2.get_yticklabels(), fontsize=11, rotation=0)
fig2.tight_layout()
fig2.savefig(FIG_DIR / "08b_error_rate.png", bbox_inches="tight")
plt.show()

## 10 · Cost / latency vs F1 trade-off

Latency reported by `meta.elapsed_sec`. We sum across the four languages → "wall time per full
benchmark run". A model on the **upper-left** of this scatter (high F1, low time) is Pareto-best.

In [ ]:
cost = (df.groupby(["model", "prompt"], observed=True)
          .agg(elapsed_sec=("elapsed_sec", "sum"),
               tokens_in=("tokens_in", "sum"),
               tokens_out=("tokens_out", "sum"))
          .join(headline[["f1", "macro_f1"]]))
display(cost.round(2))
cost.to_csv(TBL_DIR / "05_cost_latency.csv")

fig, axes = plt.subplots(1, 2, figsize=(17, 7.5))
plot = cost.reset_index()
prompt_marker = dict(zip([p for p, _ in PROMPTS], ["o", "s", "D", "^", "P", "X"]))

for ax, (xcol, xlab) in zip(axes, [("elapsed_sec", "wall time across 4 langs (sec)"),
                                    ("tokens_in",   "input tokens (sum across langs)")]):
    for _, r in plot.iterrows():
        if pd.isna(r[xcol]):
            continue
        ax.scatter(r[xcol], r["f1"],
                   marker=prompt_marker[r["prompt"]],
                   s=130, color=MODEL_PAL[r["model"]],
                   edgecolor="black", linewidth=0.5)
    ax.set_xlabel(xlab); ax.set_ylabel("micro F1")
    ax.set_title(f"F1 vs {xlab}")
    if xcol == "elapsed_sec":
        ax.set_xscale("log")

from matplotlib.lines import Line2D
mlegend = [Line2D([], [], marker="o", linestyle="", color=c, label=m,
                  markersize=10, markeredgecolor="black")
           for m, c in MODEL_PAL.items()]
plegend = [Line2D([], [], marker=mk, linestyle="", color="grey",
                  label=PROMPT_LABEL[p], markersize=9, markeredgecolor="black")
           for p, mk in prompt_marker.items()]
axes[0].legend(handles=mlegend, title="model", loc="center left",
               bbox_to_anchor=(1.02, 0.5), fontsize=9, title_fontsize=10)
axes[1].legend(handles=plegend, title="prompt", loc="center left",
               bbox_to_anchor=(1.02, 0.5), fontsize=9, title_fontsize=10)

fig.tight_layout()
fig.savefig(FIG_DIR / "09_cost_vs_f1.png", bbox_inches="tight")
plt.show()

## 11 · Statistical significance — paired bootstrap on per-record F1

For each `(model, prompt)`, we load the per-record CSV (one row per source file with TP/FP/FN),
bootstrap-resample 1000 times (paired, same record indices across systems), recompute pooled
micro F1, and report 95% CI on the **paired difference** ΔF1 = F1(B) − F1(A).
A 95% CI excluding 0 ⇒ statistically significant difference.

In [ ]:
def load_per_record(model_dir: str, prompt_v: str) -> pd.DataFrame | None:
    folder, rag_mode = PROMPT_SOURCE[prompt_v]
    parts = []
    for lang in LANGS:
        d = RUNS_DIR / model_dir / folder
        if not d.exists():
            return None
        pat = re.compile(rf"^{folder}__.+__annotated__{lang}__all__.*\.per_record\.csv$")
        cands = [p for p in d.iterdir() if pat.match(p.name)]
        if rag_mode is not None:
            cands = [p for p in cands if f"__{rag_mode}_" in p.name]
        if not cands:
            return None
        cands.sort(key=lambda p: p.stat().st_mtime)
        parts.append(pd.read_csv(cands[-1])[["sample_id", "tp", "fp", "fn"]])
    return pd.concat(parts, ignore_index=True).set_index("sample_id").sort_index()

def micro_f1_arr(tp, fp, fn):
    p = tp.sum() / max(tp.sum() + fp.sum(), 1)
    r = tp.sum() / max(tp.sum() + fn.sum(), 1)
    return 2 * p * r / (p + r) if (p + r) else 0.0

def paired_bootstrap_diff(a: pd.DataFrame, b: pd.DataFrame,
                          n_boot: int = 1000, seed: int = 42):
    common = a.index.intersection(b.index)
    a, b = a.loc[common], b.loc[common]
    rng = np.random.default_rng(seed)
    n = len(common)
    diffs = np.empty(n_boot)
    obs   = micro_f1_arr(b.tp, b.fp, b.fn) - micro_f1_arr(a.tp, a.fp, a.fn)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        diffs[i] = (micro_f1_arr(b.tp.iloc[idx], b.fp.iloc[idx], b.fn.iloc[idx]) -
                    micro_f1_arr(a.tp.iloc[idx], a.fp.iloc[idx], a.fn.iloc[idx]))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return obs, lo, hi

pr_cache = {}
for model_dir, m_label in MODELS.items():
    for p_v, _ in PROMPTS:
        pr = load_per_record(model_dir, p_v)
        if pr is not None:
            pr_cache[(m_label, p_v)] = pr
print(f"Cached per-record data for {len(pr_cache)} (model, prompt) cells.")

In [ ]:
sig_rows = []
for m_label in MODELS.values():
    base = pr_cache.get((m_label, "p1_zero_shot"))
    if base is None: continue
    for p_v, p_label in PROMPTS:
        if p_v == "p1_zero_shot": continue
        cur = pr_cache.get((m_label, p_v))
        if cur is None: continue
        d, lo, hi = paired_bootstrap_diff(base, cur)
        sig_rows.append({
            "comparison": f"{p_label} vs P1",
            "model"     : m_label,
            "ΔF1"       : round(d, 3),
            "CI 2.5%"   : round(lo, 3),
            "CI 97.5%"  : round(hi, 3),
            "sig"       : "✅" if (lo > 0 or hi < 0) else "ns",
        })
sig_within = pd.DataFrame(sig_rows)
print("Within-model: prompt vs P1 zero-shot")
display(sig_within)
sig_within.to_csv(TBL_DIR / "06_paired_bootstrap_within_model.csv", index=False)

print("\nCross-model (each at its best prompt):")
mlist = list(MODELS.values())
sig2_rows = []
for i in range(len(mlist)):
    for j in range(i + 1, len(mlist)):
        a_m, b_m = mlist[i], mlist[j]
        a = pr_cache.get((a_m, best_prompt_per_model[a_m]))
        b = pr_cache.get((b_m, best_prompt_per_model[b_m]))
        if a is None or b is None: continue
        d, lo, hi = paired_bootstrap_diff(a, b)
        sig2_rows.append({
            "A"      : f"{a_m} ({PROMPT_LABEL[best_prompt_per_model[a_m]]})",
            "B"      : f"{b_m} ({PROMPT_LABEL[best_prompt_per_model[b_m]]})",
            "ΔF1 (B−A)": round(d, 3),
            "CI 2.5%": round(lo, 3),
            "CI 97.5%": round(hi, 3),
            "sig"    : "✅" if (lo > 0 or hi < 0) else "ns",
        })
sig_cross = pd.DataFrame(sig2_rows)
display(sig_cross)
sig_cross.to_csv(TBL_DIR / "07_paired_bootstrap_cross_model.csv", index=False)

print("\nRAG ablation (dense vs random):")
sig3_rows = []
for m_label in MODELS.values():
    a = pr_cache.get((m_label, "p5_rag_random"))
    b = pr_cache.get((m_label, "p5_rag_dense"))
    if a is None or b is None: continue
    d, lo, hi = paired_bootstrap_diff(a, b)
    sig3_rows.append({
        "model": m_label, "ΔF1 (dense−random)": round(d, 3),
        "CI 2.5%": round(lo, 3), "CI 97.5%": round(hi, 3),
        "sig": "✅" if (lo > 0 or hi < 0) else "ns",
    })
display(pd.DataFrame(sig3_rows))
pd.DataFrame(sig3_rows).to_csv(TBL_DIR / "08_paired_bootstrap_rag.csv", index=False)

In [ ]:
def paired_bootstrap_pvalue(a: pd.DataFrame, b: pd.DataFrame, n_boot: int = 2000, seed: int = 42):
    common = a.index.intersection(b.index)
    a, b = a.loc[common], b.loc[common]
    rng = np.random.default_rng(seed)
    n = len(common)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        diffs[i] = (micro_f1_arr(b.tp.iloc[idx], b.fp.iloc[idx], b.fn.iloc[idx]) -
                    micro_f1_arr(a.tp.iloc[idx], a.fp.iloc[idx], a.fn.iloc[idx]))
    p_pos = (diffs <= 0).mean()
    p_neg = (diffs >= 0).mean()
    p = min(1.0, 2 * min(p_pos, p_neg))
    return p

pval_rows = []
for m_label in MODELS.values():
    base = pr_cache.get((m_label, "p1_zero_shot"))
    if base is None: continue
    for p_v, p_label in PROMPTS:
        if p_v == "p1_zero_shot": continue
        cur = pr_cache.get((m_label, p_v))
        if cur is None: continue
        p = paired_bootstrap_pvalue(base, cur)
        pval_rows.append({"comparison": f"{p_label} vs P1", "model": m_label, "p_raw": p})

pvals = pd.DataFrame(pval_rows).merge(
    sig_within, on=["comparison", "model"], how="left"
)

m = len(pvals)
order = pvals["p_raw"].sort_values().index
alpha = 0.05
holm_sig = pd.Series(False, index=pvals.index)
for rank, idx in enumerate(order, start=1):
    threshold = alpha / (m - rank + 1)
    if pvals.loc[idx, "p_raw"] < threshold:
        holm_sig.loc[idx] = True
    else:
        break
pvals["sig_holm"] = holm_sig.map({True: "significant", False: "ns"})
pvals["sig_raw_ci"] = pvals["sig"]
pvals = pvals[["comparison", "model", "ΔF1", "p_raw", "sig_raw_ci", "sig_holm"]]
pvals = pvals.sort_values("p_raw").reset_index(drop=True)

n_sig_raw = (pvals["sig_raw_ci"] == "✅").sum()
n_sig_holm = (pvals["sig_holm"] == "significant").sum()
print(f"Within-model tests: {m} total, {n_sig_raw} significant by 95% CI alone, "
      f"{n_sig_holm} significant after Holm-Bonferroni family-wise correction (alpha=0.05).")
display(pvals)
pvals.to_csv(TBL_DIR / "09_holm_bonferroni.csv", index=False)

## 10. Held-out-only RAG ablation (sibling-leakage check)

The dense index is built from the 20-file train split. About 20 of the 34 evaluated files are train members, so dense retrieval could surface *sibling* train files. Restricting the dense-vs-random paired bootstrap to the 14 held-out val+test files (and to the 8 test files) removes that confound.

In [ ]:
DATA_DIR = ROOT / "prepared_data" / "datasets" / "annotated"
split_ids = {s: [r["sample_id"] for r in json.loads((DATA_DIR / f"{s}.json").read_text())]
             for s in ("train", "val", "test")}
HELDOUT14 = sorted(split_ids["val"] + split_ids["test"])
TEST8     = sorted(split_ids["test"])
print(f"train={len(split_ids['train'])}  val={len(split_ids['val'])}  test={len(split_ids['test'])}  heldout14={len(HELDOUT14)}")

def pooled_f1(pr, ids):
    sub = pr.loc[pr.index.intersection(ids)]
    return micro_f1_arr(sub.tp, sub.fp, sub.fn), len(sub)

ho_rows = []
for m_label in MODELS.values():
    rnd = pr_cache.get((m_label, "p5_rag_random"))
    den = pr_cache.get((m_label, "p5_rag_dense"))
    if rnd is None or den is None: continue
    for subset_name, ids in (("all34", None), ("heldout14", HELDOUT14), ("test8", TEST8)):
        a = rnd if ids is None else rnd.loc[rnd.index.intersection(ids)]
        b = den if ids is None else den.loc[den.index.intersection(ids)]
        d, lo, hi = paired_bootstrap_diff(a, b)
        p = paired_bootstrap_pvalue(a, b)
        ho_rows.append({
            "model": m_label, "subset": subset_name, "n": len(a.index.intersection(b.index)),
            "dense_f1": round(micro_f1_arr(b.tp, b.fp, b.fn), 3),
            "random_f1": round(micro_f1_arr(a.tp, a.fp, a.fn), 3),
            "ΔF1 (dense−random)": round(d, 3), "CI 2.5%": round(lo, 3), "CI 97.5%": round(hi, 3),
            "p_raw": round(p, 4), "sig": "✅" if (lo > 0 or hi < 0) else "ns",
        })
rag_heldout = pd.DataFrame(ho_rows)
display(rag_heldout)
rag_heldout.to_csv(TBL_DIR / "10_rag_ablation_heldout.csv", index=False)

## 11. Holm–Bonferroni within each test family

Three families are corrected separately: 45 within-model prompt-vs-P1 tests (already in cell 27), the 8 cross-model best-vs-best tests against the overall leader, and the 9 dense-vs-random RAG tests.

In [ ]:
def holm(pvals_series, alpha=0.05):
    m = len(pvals_series); order = pvals_series.sort_values().index
    out = pd.Series(False, index=pvals_series.index)
    for rank, idx in enumerate(order, start=1):
        if pvals_series.loc[idx] < alpha / (m - rank + 1): out.loc[idx] = True
        else: break
    return out

leader = max(MODELS.values(), key=lambda m: micro_f1_arr(*[pr_cache[(m, best_prompt_per_model[m])][c] for c in ("tp","fp","fn")]))
print("Leader:", leader, PROMPT_LABEL[best_prompt_per_model[leader]])
lead_pr = pr_cache[(leader, best_prompt_per_model[leader])]

cross_rows = []
for m_label in MODELS.values():
    if m_label == leader: continue
    other = pr_cache[(m_label, best_prompt_per_model[m_label])]
    d, lo, hi = paired_bootstrap_diff(lead_pr, other)
    p = paired_bootstrap_pvalue(lead_pr, other)
    cross_rows.append({"family": "cross-model", "comparison": f"{m_label} ({PROMPT_LABEL[best_prompt_per_model[m_label]]}) vs {leader}",
                       "model": m_label, "ΔF1": round(d, 3), "CI 2.5%": round(lo, 3), "CI 97.5%": round(hi, 3),
                       "p_raw": p, "sig_raw_ci": "✅" if (lo > 0 or hi < 0) else "ns"})
cross_fam = pd.DataFrame(cross_rows)
cross_fam["sig_holm"] = holm(cross_fam["p_raw"]).map({True: "significant", False: "ns"})

rag_rows = []
for m_label in MODELS.values():
    a = pr_cache.get((m_label, "p5_rag_random")); b = pr_cache.get((m_label, "p5_rag_dense"))
    if a is None or b is None: continue
    d, lo, hi = paired_bootstrap_diff(a, b); p = paired_bootstrap_pvalue(a, b)
    rag_rows.append({"family": "rag", "comparison": "dense vs random", "model": m_label, "ΔF1": round(d, 3),
                     "CI 2.5%": round(lo, 3), "CI 97.5%": round(hi, 3), "p_raw": p,
                     "sig_raw_ci": "✅" if (lo > 0 or hi < 0) else "ns"})
rag_fam = pd.DataFrame(rag_rows)
rag_fam["sig_holm"] = holm(rag_fam["p_raw"]).map({True: "significant", False: "ns"})

within_fam = pvals.copy(); within_fam.insert(0, "family", "within-model")
holm_all = pd.concat([within_fam, cross_fam, rag_fam], ignore_index=True)
for fam, g in holm_all.groupby("family"):
    print(f"{fam:13s} tests={len(g):2d}  raw-sig={(g.sig_raw_ci=='✅').sum():2d}  holm-sig={(g.sig_holm=='significant').sum():2d}")
display(cross_fam); display(rag_fam)
holm_all.to_csv(TBL_DIR / "09_holm_bonferroni.csv", index=False)

## 12. P4 failure breakdown: truncation vs. malformed JSON, and the output-token cap actually used

The `truncated` flag in the runner is a heuristic that overlaps `parse_error`. Here each P4 parse failure is reclassified from the raw response text, and cross-tabulated with the `num_predict` recorded in each run's metadata.

In [ ]:
def classify_raw(raw: str, out_tokens=None, cap=None):
    if not raw or not raw.strip(): return "request_failed"
    if out_tokens is not None and cap and out_tokens >= cap: return "truncated"
    s = raw.strip()
    if "<answer>" in s:
        s = s.split("<answer>", 1)[1]
        if "</answer>" not in s: return "truncated"
        s = s.split("</answer>", 1)[0].strip()
    if s.count("{") > s.count("}") or s.rstrip().endswith((",", '"', ":")): return "truncated"
    try:
        json.loads(s[s.find("{"): s.rfind("}") + 1]); return "parsed_late"
    except Exception:
        return "malformed"

p4_rows = []; cfg_rows = []
for model_dir, m_label in MODELS.items():
    for p_v, _ in PROMPTS:
        folder, rag_mode = PROMPT_SOURCE[p_v]
        d = RUNS_DIR / model_dir / folder
        if not d.exists(): continue
        for lang in LANGS:
            pat = re.compile(rf"^{folder}__.+__annotated__{lang}__all__.*\.metrics\.json$")
            cands = [p for p in d.iterdir() if pat.match(p.name)]
            if rag_mode: cands = [p for p in cands if f"__{rag_mode}_" in p.name]
            if not cands: continue
            cands.sort(key=lambda p: p.stat().st_mtime); mf = cands[-1]
            meta = json.loads(mf.read_text())["meta"]
            cfg_rows.append({"model": m_label, "prompt": p_v, "language": lang, "num_predict": meta.get("num_predict"),
                             "num_ctx": meta.get("num_ctx"), "timestamp": meta.get("timestamp"), "provider": meta.get("provider")})
            if p_v == "p4_self_verify":
                preds = json.loads(mf.with_name(mf.name.replace(".metrics.json", ".predictions.json")).read_text())
                for r in preds:
                    cls = classify_raw(r.get("raw", ""), r.get("output_tokens"), meta.get("num_predict")) if r.get("parse_error") else "ok"
                    at_cap = (r.get("output_tokens") or 0) >= 0.95 * (meta.get("num_predict") or 1e9)
                    p4_rows.append({"model": m_label, "language": lang, "sample_id": r["sample_id"], "parse_error": bool(r.get("parse_error")),
                                    "class": cls, "output_tokens": r.get("output_tokens"), "num_predict": meta.get("num_predict"), "at_cap": at_cap})
cfg = pd.DataFrame(cfg_rows); cfg.to_csv(TBL_DIR / "12_run_config_audit.csv", index=False)
print("num_predict by model:"); display(cfg.groupby("model", observed=True)["num_predict"].unique())

p4 = pd.DataFrame(p4_rows)
p4_sum = (p4.groupby("model", observed=True)
            .agg(n=("sample_id", "size"), parse_errors=("parse_error", "sum"),
                 truncated=("class", lambda s: (s == "truncated").sum()),
                 malformed=("class", lambda s: (s == "malformed").sum()),
                 request_failed=("class", lambda s: (s == "request_failed").sum()),
                 parsed_late=("class", lambda s: (s == "parsed_late").sum()),
                 at_cap=("at_cap", "sum"), num_predict=("num_predict", "first"))
            .reset_index())
p4_sum["parse_error_rate"] = (p4_sum.parse_errors / p4_sum.n).round(3)
p4_sum["pct_truncated_of_errors"] = (p4_sum.truncated / p4_sum.parse_errors.replace(0, np.nan)).round(3)
display(p4_sum)
p4_sum.to_csv(TBL_DIR / "11_p4_failure_breakdown.csv", index=False)

## 13. Run-to-run noise: the one same-configuration repeat available

Nemotron Nano 9B v2's random-retrieval arm was run twice (archived pilot vs. live run). The paired ΔF1 between them is the only empirical noise-floor estimate in the data.

In [ ]:
ARCH = ROOT / "results" / "_archive" / "superseded_k2_pilot"
def load_arch_random(model_tag: str):
    parts = []
    for lang in LANGS:
        cands = sorted(ARCH.rglob(f"p5_rag__{model_tag}__annotated__{lang}__all__random_k2__*.per_record.csv"))
        if not cands: return None
        parts.append(pd.read_csv(cands[-1])[["sample_id", "tp", "fp", "fn"]])
    return pd.concat(parts, ignore_index=True).set_index("sample_id").sort_index()

noise_rows = []
arch = load_arch_random("nvidia.nemotron-nano-9b-v2")
live = pr_cache.get(("Nemotron Nano 9B v2", "p5_rag_random"))
if arch is not None and live is not None:
    d, lo, hi = paired_bootstrap_diff(arch, live)
    noise_rows.append({"model": "Nemotron Nano 9B v2", "arm": "P5 random (k=2)",
                       "f1_run1_archived": round(micro_f1_arr(arch.tp, arch.fp, arch.fn), 3),
                       "f1_run2_live": round(micro_f1_arr(live.tp, live.fp, live.fn), 3),
                       "ΔF1 (run2−run1)": round(d, 3), "CI 2.5%": round(lo, 3), "CI 97.5%": round(hi, 3),
                       "n_records": len(arch.index.intersection(live.index)),
                       "records_changed": int((arch.loc[live.index.intersection(arch.index)][["tp","fp","fn"]].values !=
                                               live.loc[live.index.intersection(arch.index)][["tp","fp","fn"]].values).any(axis=1).sum())})
noise = pd.DataFrame(noise_rows); display(noise)
noise.to_csv(TBL_DIR / "13_repeat_run_noise.csv", index=False)

## 12 · Discussion & paper-ready key findings

The cell below auto-generates a textual summary that updates whenever the data is regenerated.
Use it as the **scaffold** for the paper's Results / Discussion sections; rephrase as needed.

In [ ]:
from textwrap import dedent

best_model, best_prompt = headline["f1"].idxmax()
best_f1 = headline["f1"].max()

lift = delta.drop(columns=["p1_zero_shot"]).mean(axis=0).sort_values(ascending=False)

rag_lift = (rag_pooled["dense"] - rag_pooled["random"]).round(3).dropna()
rag_helpers = rag_lift[rag_lift > 0.01].index.tolist()
rag_neutral = rag_lift[rag_lift.abs() <= 0.01].index.tolist()

hardest = (ps_pivot[list(MODELS.values())].mean(axis=1)
           .where(ps_pivot["support"] >= 10).dropna()
           .sort_values().head(5).round(2))
easiest = (ps_pivot[list(MODELS.values())].mean(axis=1)
           .where(ps_pivot["support"] >= 10).dropna()
           .sort_values(ascending=False).head(5).round(2))

lang_rank = lang_mean["mean"].sort_values(ascending=False)

summary = f"""
# Key Findings (auto-generated)

## Headline result
- **Best system:** *{best_model}* with **{PROMPT_LABEL[best_prompt]}** → micro F1 = **{best_f1:.3f}**
  (pooled across Java, Python, JavaScript, C++).
- Per-model best prompts: """ + "; ".join(
    f"**{m}** → {PROMPT_LABEL[p]}" for m, p in best_prompt_per_model.items()
) + f""".

## RQ1 — Does few-shot beat zero-shot?
- Mean lift of P2 few-shot over P1 across {len(MODELS)} models: **{lift.get('p2_few_shot', 0):+.3f}** F1.
- Few-shot improves all models: { 'yes' if (delta['p2_few_shot'] > 0).all() else 'mixed' }.

## RQ2 — Does structured / self-correction reasoning help?
- P3 taxonomy-tree mean lift: **{lift.get('p3_taxonomy_tree', 0):+.3f}** F1.
- P4 self-verify mean lift:    **{lift.get('p4_self_verify', 0):+.3f}** F1.
- Take-away: structured reasoning tends to {'help' if lift.get('p3_taxonomy_tree', 0) > 0 else 'hurt'};
  self-verify is {'consistently positive' if (delta['p4_self_verify'] > 0).all()
                                      else 'inconsistent across models'}.

## RQ3 — Does RAG content matter?
- Dense (k=3) − random (k=2) per model (random arm run for all nine models):
""" + "\n".join(f"  - {m}: {rag_lift[m]:+.3f}" for m in rag_lift.index) + f"""
- Models that benefit from real retrieval: **{', '.join(rag_helpers) or 'none'}**.
- Models where dense ≈ random: **{', '.join(rag_neutral) or 'none'}** → suggests the smell taxonomy
  in the system prompt already supplies most of the signal these models can exploit; retrieval mainly
  acts as a {'\"prompt expander\"' if rag_neutral else 'real evidence channel'}.

## Per-smell — easiest vs hardest
- **Easiest (highest mean F1, support ≥10):** {', '.join(f'{s} ({v:.2f})' for s, v in easiest.items())}
- **Hardest (lowest mean F1, support ≥10):**  {', '.join(f'{s} ({v:.2f})' for s, v in hardest.items())}

## Language difficulty
""" + "\n".join(f"- {l}: mean F1 = {v:.3f}" for l, v in lang_rank.items()) + f"""

## Cost / latency
- Best F1-per-second is achieved at *{cost.f1.idxmax()}* (F1={cost.f1.max():.3f},
  total elapsed {cost.elapsed_sec.loc[cost.f1.idxmax()]:.0f} s across 4 langs).

## Recommended setup for production
- **Quality-first:** *{best_model} + {PROMPT_LABEL[best_prompt]}*.
- **Latency-first:** the model with highest F1 in P1 zero-shot — `{f1_pivot['p1_zero_shot'].idxmax()}`
  (F1 = {f1_pivot['p1_zero_shot'].max():.3f}); skips few-shot/RAG context overhead.
- **Open question for future work:** the long tail of low-support smells (Lazy Class, Comments,
  Data Class, Feature Envy) is unstable across runs — collect more annotations before drawing
  conclusions on those classes.
"""

print(summary)
(TBL_DIR / "00_key_findings.md").write_text(summary)
print(f"\nSaved → {TBL_DIR / '00_key_findings.md'}")

---

### Outputs

All tables saved to `results/tables/final_comparative/`:
- `00_key_findings.md` — paper-ready summary
- `01_headline_micro_macro.csv` — headline table
- `02_rag_ablation_per_lang.csv` — dense vs random per language
- `03_per_smell_f1_best_prompt.csv` — per-smell F1 (best prompt per model)
- `04_error_mix.csv` — TP/FP/FN + parse-error rate
- `05_cost_latency.csv` — wall time and tokens vs F1
- `06_paired_bootstrap_within_model.csv` — significance of each prompt vs P1
- `07_paired_bootstrap_cross_model.csv` — significance of model A vs B (best prompt each)
- `08_paired_bootstrap_rag.csv` — significance of dense vs random RAG

All figures saved to `results/figures/final_comparative/`.

> **Reproducibility:** every cell reads only from `results/llm_runs/<model>/<prompt>/*.metrics.json`
> and `*.per_record.csv`. Re-run any benchmark and re-execute the notebook top-to-bottom to
> refresh tables, figures, and the auto-generated discussion.